In [1]:
import pickle

import matplotlib.pyplot as plt
import numpy as np
from scipy.special import kl_div
from scipy.stats import anderson, kurtosis, wasserstein_distance
from tqdm import tqdm
from tqdm.contrib.concurrent import process_map

In [2]:
def generate_random_pairs(num_pairs, input_size):
    """
    @param num_pairs is the number of pairs of input samples to be selected
    @param input_size is the number of input samples
    @returns two numpy arrays of indices, where (indices1[i], indices2[i]) are
                the indices for pair i
    """

    indices = np.random.choice(input_size, size=num_pairs * 2, replace=False)

    # shuffle the indices
    np.random.shuffle(indices)

    indices1 = indices[:num_pairs]
    indices2 = indices[num_pairs:]

    return indices1, indices2

In [ ]:
# Compute N_x (maximum L2 norm between input pairs)
def calculate_Nx(inputs, indices1, indices2):
    """
    @param inputs a flattened inputs numpy array, in which the shape is (number of tokens, token dimension)
    @param indices1 see spec for function generate_random_pairs
    @param indices2 see spec for function generate_random_pairs
    @returns a numpy array of shape (number of pairs, ) in which the value at index i corresponds to the
                L2 norm difference of input pair i; and the maximum L2 norm
    """
    # pairwise_distances = pdist(inputs) # this should be used in the case that you want to use all possible input pairs
    pairwise_distances = np.linalg.norm(inputs[indices1] - inputs[indices2], axis=1) # shape is (num_pairs, )
    N_x = np.max(pairwise_distances)  # The maximum L2 norm
    return pairwise_distances, N_x

# Comput N_y (median L2 norm between output pairs)
def calculate_Ny(outputs, indices1, indices2):
    """
    @param outputs is a flattened outputs numpy array with shape (number of tokens, );
            the value at index i corresponds to the scalar dot product between the neuron and input sample i
    @param indices1 see spec for function generate_random_pairs
    @param indices2 see spec for function generate_random_pairs
    @returns the median L2 norm difference between output pairs specified by the indices
    """
    pairwise_distances = np.abs(outputs[indices1] - outputs[indices2]) # just calculates the absolute value since outputs are scalar values
    N_y = np.median(pairwise_distances)
    return N_y

In [4]:
# Helper function to calculate output pairwise distances for a given neuron
def calculate_output_pairwise_distances(neuron_outputs, indices1, indices2):
    """
    @param neuron_outputs is a flattened numpy array of the scalar outputs for a given neuron
    @returns: a numpy array of size (len(indices1), ) because len(indices1) is the number of pairs;
              L2 norm between outputs associated with pair i is at index i
    """
    output_l2_norms = []
    num_pairs = len(indices1)

    for pair_index in range(num_pairs):
        i = indices1[pair_index]
        j = indices2[pair_index]
        l2_norm = abs(neuron_outputs[i] - neuron_outputs[j])
        output_l2_norms.append(l2_norm)

    return np.array(output_l2_norms)

In [5]:
# Compute Mapping Difficulty (MD) for one neuron
def calculate_MD(inputs_pairwise_distances, N_x, neuron_outputs, indices1, indices2):
    """
    @param inputs_pairwise_distances: has shape (num_pairs, ) and contains L2 norm for pair i at index i
    @param N_x: maximum norm between input pairs
    @param neuron_outputs: is a numpy array of the scalar outputs for a given neuron
    """
    # initialize variables
    md_sum = 0
    num_pairs = len(indices1)
    N_y = calculate_Ny(neuron_outputs, indices1, indices2)
    output_pairwise_distances = calculate_output_pairwise_distances(neuron_outputs, indices1, indices2)

    # loop through all the pairs of indices
    for pair_index in range(num_pairs):
        numerator = output_pairwise_distances[pair_index]/N_y
        denominator = inputs_pairwise_distances[pair_index]/N_x
        md_sum += numerator/denominator

    # return average
    return md_sum/num_pairs

In [ ]:
# multi-threading
def acquire_MD(neuron_outputs):
    return calculate_MD(inputs_pairwise_distances, N_x, neuron_outputs, indices1, indices2)


In [ ]:
# input data and dense data acquired from IO_collector.ipynb

input_data = np.load("input_data.npy")
dense_data = np.load("dense_data.npy")

input_size = input_data.shape[0] * input_data.shape[1] # number of input tokens
num_pairs = 50000 # how many pairs to sample for MD calculation

indices1, indices2 = generate_random_pairs(num_pairs, input_size)

inputs_pairwise_distances, N_x = calculate_Nx(input_data.reshape(-1, input_data.shape[-1]), indices1, indices2) # inputs are same for all neurons

In [ ]:
MDs = process_map(acquire_MD, dense_data.reshape(-1, dense_data.shape[-1]).T, max_workers=32, chunksize=1)